# RANDOMNESS SET

In [6]:
# from alignn import pretrained
# pretrained.get_all_models()

import torch, os
import random
import numpy as np
import pandas as pd
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

# Alternatively, you can directly set the device
torch.cuda.set_device(0)  # Replace "1" with the index of the GPU you want to use

def set_seed():
    os.environ["WANDB_ANONYMOUS"] = "must"
    random_seed = 42
    random.seed(random_seed)
    torch.manual_seed(random_seed)
    np.random.seed(random_seed)
    torch.cuda.manual_seed_all(random_seed)
    try:
        import torch_xla.core.xla_model as xm
        xm.set_rng_state(random_seed)
    except ImportError:
        pass
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(random_seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = str(":4096:8")
    torch.use_deterministic_algorithms(True)

set_seed()

# ALIGNN Models

In [2]:
# Models in Alignn
from alignn import pretrained

models = ["jv_mbj_bandgap_alignn", "jv_optb88vdw_bandgap_alignn", "jv_supercon_tc_alignn"]

align_model = pretrained.get_figshare_model(models[1]) 

align_model

Using chk file jv_optb88vdw_bandgap_alignn/checkpoint_300.pt from  ['jv_optb88vdw_bandgap_alignn/checkpoint_300.pt']
Path /home/jipengsun/.conda/envs/llm/lib/python3.10/site-packages/alignn/jv_optb88vdw_bandgap_alignn.zip
Config /home/jipengsun/LLM_Atom_Gen/jv_optb88vdw_bandgap_alignn/config.json


/home/jipengsun/.conda/envs/llm/lib/python3.10/site-packages/alignn/pretrained.py:292: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(filenam

ALIGNN(
  (atom_embedding): MLPLayer(
    (layer): Sequential(
      (0): Linear(in_features=92, out_features=256, bias=True)
      (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU()
    )
  )
  (edge_embedding): Sequential(
    (0): RBFExpansion()
    (1): MLPLayer(
      (layer): Sequential(
        (0): Linear(in_features=80, out_features=64, bias=True)
        (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): SiLU()
      )
    )
    (2): MLPLayer(
      (layer): Sequential(
        (0): Linear(in_features=64, out_features=256, bias=True)
        (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): SiLU()
      )
    )
  )
  (angle_embedding): Sequential(
    (0): RBFExpansion()
    (1): MLPLayer(
      (layer): Sequential(
        (0): Linear(in_features=40, out_features=64, bias=True)
        (1): BatchNorm1d(64, eps=1e-05, momentum=0.

In [ ]:
from models import load_model, get_model, get_trainer, save_model
import os
from unsloth import FastLanguageModel
from datasets import load_dataset
import pandas as pd 
from utils import eval_prompts
from sample_funcs import parse_fn
from transformers import TextStreamer


llm_path = "./0_models_tc_supercon"
fourbit_models = [
        "unsloth/tinyllama-chat", #X
        "unsloth/mistral-7b-bnb-4bit", #X
        "unsloth/gemma-7b-bnb-4bit", #X
        "unsloth/llama-3-8b-bnb-4bit", #X 
]  # More models at https://huggingface.co/unsloth

for model_idx, models in enumerate(fourbit_models):

        # LLM Models

        fourbit_model = fourbit_models[model_idx]   
        print("Models Name", fourbit_model)
        llm_model, llm_tokenizer = FastLanguageModel.from_pretrained(os.path.join(llm_path, fourbit_model.split("/")[1])) 
        FastLanguageModel.for_inference(llm_model)  # Enable native 2x faster inference

        # Data Loading
        
        data_test = load_dataset("json", data_files="./data/alpaca_"+ "Tc_supercon" +"_test.json", split="train") # #optb88vdw_bandgap  mbj_bandgap Tc_supercon
        data_test = data_test.add_column("prompt", eval_prompts(data_test))
        df = pd.DataFrame(data_test)

        # LLM OUTPUT GENERATION
        
        for idx, prompt in enumerate(data_test["prompt"]):
                print(idx)
                batch = llm_tokenizer(prompt, return_tensors="pt")
                batch = {k: v.cuda() for k, v in batch.items()}
                generate_ids = llm_model.generate(
                                **batch,
                                do_sample=False,
                                max_new_tokens=4096,
                                pad_token_id=llm_tokenizer.eos_token_id,
                                use_cache=True)

                gen_strs = llm_tokenizer.batch_decode(
                                generate_ids, 
                                skip_special_tokens=True, 
                                clean_up_tokenization_spaces=True
                                )
                try:
                        material_str = gen_strs[0].replace(prompt, "")
                        cif_str = parse_fn(material_str)
                        df.loc[idx, 'gen_material_str'] = material_str
                        df.loc[idx, 'gen_material_cif'] = cif_str
                        df.loc[idx, 'orj_material_cif'] = parse_fn(data_test["response"][idx])
                except Exception as e:
                        print(e)
                #df
        df.to_csv(f"./0_gen_tc_supercon/{fourbit_model.split('/')[1]}_generated_samples.csv", index=False)
        del(df)  

# Measure the Performance

In [3]:
from pymatgen.analysis.structure_matcher import StructureMatcher
matcher = StructureMatcher(stol=0.5, angle_tol=10, ltol=0.3)
from jarvis.core.atoms import Atoms
from pymatgen.core.structure import Structure
import jarvis 

In [10]:
# Read the files
fourbit_models = [
        #"unsloth/tinyllama-chat", #X
        "unsloth/mistral-7b-bnb-4bit", #X
        "unsloth/gemma-7b-bnb-4bit", #X
        "unsloth/llama-3-8b-bnb-4bit", #X 
]  # More models at https://huggingface.co/unsloth

# TC_Supercon

In [ ]:
for idx, name in enumerate(fourbit_models):

        #Read the model
        fourbit_model = fourbit_models[idx]
        path = f"./0_gen_tc_supercon/{fourbit_model.split('/')[1]}_generated_samples.csv"
        df = pd.read_csv(path)
        print(name)
        #Look at each models
        for jdx, nname in df.iterrows():

                try:
                        str_pred = Structure.from_str(df["gen_material_cif"][jdx], fmt="cif")
                        str_tar = Structure.from_str(df["orj_material_cif"][jdx], fmt="cif")
                        atoms_pred = jarvis.core.atoms.pmg_to_atoms(str_pred).pymatgen_converter()
                        atoms_tar = jarvis.core.atoms.pmg_to_atoms(str_tar).pymatgen_converter()
                        rms_dist = matcher.get_rms_anonymous(atoms_pred, atoms_tar)
                        df.loc[jdx, 'rms_dist'] = rms_dist[0]
                        
                        try:
                                df.loc[jdx, 'orj_prop_val'] = float(nname["input"].split("value is ")[1][:-1])
                        except Exception as e:
                                print('orj_prop_val')
                                df.loc[jdx, 'orj_prop_val'] = None

                        try:
                                out_data_pred = pretrained.get_prediction(
                                        model_name="jv_supercon_tc_alignn",
                                        atoms=jarvis.core.atoms.pmg_to_atoms(str_pred),
                                )
                                df.loc[jdx, 'out_data_pred'] = out_data_pred[0]
                        except Exception as e:
                                print('out_data_pred')
                                df.loc[jdx, 'out_data_pred'] = None    

                        try:           
                                out_data_tar = pretrained.get_prediction(
                                        model_name="jv_supercon_tc_alignn",
                                        atoms=jarvis.core.atoms.pmg_to_atoms(str_tar),
                                )
                                df.loc[jdx, 'out_data_tar'] = out_data_tar[0]

                        except Exception as e:
                                print('out_data_tar')
                                df.loc[jdx, 'out_data_tar'] = None  
                        

                except Exception as e:
                        print(e)
                
        df.to_csv(f"./0_gen_tc_supercon/{fourbit_model.split('/')[1]}_generated_samples_updated.csv", index=False)
        del(df)  
        
                

# optb88vdw_bandgap

In [12]:
for idx, name in enumerate(fourbit_models):

        #Read the model
        fourbit_model = fourbit_models[idx]
        path = f"./1_gen_optb88vdw_bandgap/{fourbit_model.split('/')[1]}_generated_samples.csv"
        df = pd.read_csv(path)

        #Look at each models
        for jdx, nname in df.iterrows():
                
                if jdx == 300:
                        break
                try:
                        str_pred = Structure.from_str(df["gen_material_cif"][jdx], fmt="cif")
                        str_tar = Structure.from_str(df["orj_material_cif"][jdx], fmt="cif")
                        atoms_pred = jarvis.core.atoms.pmg_to_atoms(str_pred).pymatgen_converter()
                        atoms_tar = jarvis.core.atoms.pmg_to_atoms(str_tar).pymatgen_converter()
                        rms_dist = matcher.get_rms_anonymous(atoms_pred, atoms_tar)
                        df.loc[jdx, 'rms_dist'] = rms_dist[0]
                        
                        try:
                                df.loc[jdx, 'orj_prop_val'] = float(nname["input"].split("value is ")[1][:-1])
                        except Exception as e:
                                print('orj_prop_val')
                                df.loc[jdx, 'orj_prop_val'] = None

                        try:
                                out_data_pred = pretrained.get_prediction(
                                        model_name="jv_optb88vdw_bandgap_alignn",
                                        atoms=jarvis.core.atoms.pmg_to_atoms(str_pred),
                                )
                                df.loc[jdx, 'out_data_pred'] = out_data_pred[0]
                        except Exception as e:
                                print('out_data_pred')
                                df.loc[jdx, 'out_data_pred'] = None    

                        try:           
                                out_data_tar = pretrained.get_prediction(
                                        model_name="jv_optb88vdw_bandgap_alignn",
                                        atoms=jarvis.core.atoms.pmg_to_atoms(str_tar),
                                )
                                df.loc[jdx, 'out_data_tar'] = out_data_tar[0]

                        except Exception as e:
                                print('out_data_tar')
                                df.loc[jdx, 'out_data_tar'] = None  
                        

                except Exception as e:
                        print(e)
                

        df.to_csv(f"./1_gen_optb88vdw_bandgap/{fourbit_model.split('/')[1]}_generated_samples_updated.csv", index=False)
        del(df)  
        
                

Using chk file jv_optb88vdw_bandgap_alignn/checkpoint_300.pt from  ['jv_optb88vdw_bandgap_alignn/checkpoint_300.pt']
Path /home/jipengsun/.conda/envs/llm/lib/python3.10/site-packages/alignn/jv_optb88vdw_bandgap_alignn.zip
Config /home/jipengsun/LLM_Atom_Gen/jv_optb88vdw_bandgap_alignn/config.json
out_data_pred
Using chk file jv_optb88vdw_bandgap_alignn/checkpoint_300.pt from  ['jv_optb88vdw_bandgap_alignn/checkpoint_300.pt']
Path /home/jipengsun/.conda/envs/llm/lib/python3.10/site-packages/alignn/jv_optb88vdw_bandgap_alignn.zip
Config /home/jipengsun/LLM_Atom_Gen/jv_optb88vdw_bandgap_alignn/config.json
out_data_tar
Using chk file jv_optb88vdw_bandgap_alignn/checkpoint_300.pt from  ['jv_optb88vdw_bandgap_alignn/checkpoint_300.pt']
Path /home/jipengsun/.conda/envs/llm/lib/python3.10/site-packages/alignn/jv_optb88vdw_bandgap_alignn.zip
Config /home/jipengsun/LLM_Atom_Gen/jv_optb88vdw_bandgap_alignn/config.json
out_data_pred
Using chk file jv_optb88vdw_bandgap_alignn/checkpoint_300.pt from

KeyboardInterrupt: 

# mbj_bandgap

In [ ]:
for idx, name in enumerate(fourbit_models):

        #Read the model
        fourbit_model = fourbit_models[idx]
        path = f"./2_gen_mbj_bandgap/{fourbit_model.split('/')[1]}_generated_samples.csv"
        df = pd.read_csv(path)

        #Look at each models
        for jdx, nname in df.iterrows():
                
                if jdx == 300:
                        break
                try:
                        str_pred = Structure.from_str(df["gen_material_cif"][jdx], fmt="cif")
                        str_tar = Structure.from_str(df["orj_material_cif"][jdx], fmt="cif")
                        atoms_pred = jarvis.core.atoms.pmg_to_atoms(str_pred).pymatgen_converter()
                        atoms_tar = jarvis.core.atoms.pmg_to_atoms(str_tar).pymatgen_converter()
                        rms_dist = matcher.get_rms_anonymous(atoms_pred, atoms_tar)
                        df.loc[jdx, 'rms_dist'] = rms_dist[0]
                        
                        try:
                                df.loc[jdx, 'orj_prop_val'] = float(nname["input"].split("value is ")[1][:-1])
                        except Exception as e:
                                print('orj_prop_val')
                                df.loc[jdx, 'orj_prop_val'] = None

                        try:
                                out_data_pred = pretrained.get_prediction(
                                        model_name="jv_mbj_bandgap_alignn",
                                        atoms=jarvis.core.atoms.pmg_to_atoms(str_pred),
                                )
                                df.loc[jdx, 'out_data_pred'] = out_data_pred[0]
                        except Exception as e:
                                print('out_data_pred')
                                df.loc[jdx, 'out_data_pred'] = None    

                        try:           
                                out_data_tar = pretrained.get_prediction(
                                        model_name="jv_mbj_bandgap_alignn",
                                        atoms=jarvis.core.atoms.pmg_to_atoms(str_tar),
                                )
                                df.loc[jdx, 'out_data_tar'] = out_data_tar[0]

                        except Exception as e:
                                print('out_data_tar')
                                df.loc[jdx, 'out_data_tar'] = None  
                        

                except Exception as e:
                        print(e)
                

        df.to_csv(f"./2_gen_mbj_bandgap/{fourbit_model.split('/')[1]}_generated_samples_updated.csv", index=False)
        del(df)  
        
                